# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nioshis-cmd/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
import duckdb
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    "CREATE SECRET (TYPE HUGGINGFACE, TOKEN ?)",
    [hf_token]
)

print("Hugging Face connection ready.")

Hugging Face connection ready.


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My rule: I will prioritize pages that have relatively high search visibility (gsc_impressions) but weaker average search position (gsc_avg_position). These pages may have enough visibility to be worth further review.

Reason codes: HIGH_VOLUME_WEAK_POSITION for pages matching this pattern, and LOW_PRIORITY for pages that do not.

In [9]:
query = """
SELECT
    CASE
        WHEN gsc_impressions = 0 THEN '0'
        WHEN gsc_impressions < 100 THEN '1-99'
        WHEN gsc_impressions < 500 THEN '100-499'
        ELSE '500+'
    END AS impressions_bucket,
    COUNT(*) AS n
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
GROUP BY impressions_bucket
ORDER BY impressions_bucket
"""

display(con.sql(query).df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,impressions_bucket,n
0,0,6230317
1,1-99,2972453
2,100-499,537157
3,500+,101451


Signal 1 verdict: CONFIRMED. Impressions provide a useful measure of search visibility and can help identify pages with enough visibility to justify further review.

In [10]:
query = """
SELECT
    CASE
        WHEN gsc_avg_position < 5 THEN 'Top 5'
        WHEN gsc_avg_position < 10 THEN '5-10'
        WHEN gsc_avg_position < 20 THEN '10-20'
        ELSE '20+'
    END AS position_bucket,
    COUNT(*) AS n
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_avg_position IS NOT NULL
GROUP BY position_bucket
ORDER BY position_bucket
"""

display(con.sql(query).df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,position_bucket,n
0,10-20,543782
1,20+,912338
2,5-10,932273
3,Top 5,1222668


Signal 2 verdict: CONFIRMED. Average position provides a useful visibility signal because pages occur across different position ranges. A weaker position can be a reason to prioritize a page when it also has meaningful impressions.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [11]:
query = """
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS impressions,
    SUM(gsc_clicks) AS clicks,
    AVG(gsc_avg_position) AS avg_position
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
GROUP BY client_hash_id, content_hash_id
"""

queue = con.sql(query).df()

queue = queue[
    (queue["impressions"] > 0) &
    (queue["avg_position"].notna())
].copy()

queue["impressions_rank"] = queue["impressions"].rank(pct=True)
queue["position_rank"] = queue["avg_position"].rank(pct=True)

queue["score"] = (
    queue["impressions_rank"] * queue["position_rank"]
)

median_impressions = queue["impressions"].median()
median_position = queue["avg_position"].median()

queue["reason_code"] = "LOW_PRIORITY"
queue["action"] = "MONITOR"

mask = (
    (queue["impressions"] >= median_impressions) &
    (queue["avg_position"] >= median_position)
)

queue.loc[mask, "reason_code"] = "HIGH_VOLUME_WEAK_POSITION"
queue.loc[mask, "action"] = "REVIEW"

queue = queue.sort_values("score", ascending=False).reset_index(drop=True)
queue["rank"] = range(1, len(queue) + 1)

queue = queue[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "impressions",
        "clicks",
        "avg_position",
        "score",
        "reason_code",
        "action"
    ]
]

display(queue.head(20))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rank,client_hash_id,content_hash_id,impressions,clicks,avg_position,score,reason_code,action
0,1,client_23a62021009f63c4,content_6aa54d6bbdbf6f24,39405.0,3.0,54.384283,0.942041,HIGH_VOLUME_WEAK_POSITION,REVIEW
1,2,client_23a62021009f63c4,content_1ff3c48911f11e70,27684.0,2.0,55.147128,0.940694,HIGH_VOLUME_WEAK_POSITION,REVIEW
2,3,client_fef1a8f436438636,content_1c47c13983830602,20904.0,7.0,56.597228,0.939987,HIGH_VOLUME_WEAK_POSITION,REVIEW
3,4,client_23a62021009f63c4,content_2de9a39d3482a269,31664.0,6.0,53.143951,0.937376,HIGH_VOLUME_WEAK_POSITION,REVIEW
4,5,client_3197e6291363b4db,content_65b8a4998e633d89,8905.0,0.0,68.385294,0.935876,HIGH_VOLUME_WEAK_POSITION,REVIEW
5,6,client_e547b89c05043229,content_e2ab9def8659a580,8813.0,1.0,67.894995,0.934554,HIGH_VOLUME_WEAK_POSITION,REVIEW
6,7,client_e547b89c05043229,content_ba98b9ba325cb149,12292.0,5.0,59.867808,0.933224,HIGH_VOLUME_WEAK_POSITION,REVIEW
7,8,client_23a62021009f63c4,content_6530fa9d297c46eb,9302.0,1.0,65.017571,0.931965,HIGH_VOLUME_WEAK_POSITION,REVIEW
8,9,client_23a62021009f63c4,content_736f0fed2392ab68,16179.0,1.0,55.441210,0.931725,HIGH_VOLUME_WEAK_POSITION,REVIEW
9,10,client_e547b89c05043229,content_835b45001eda8e4c,6775.0,1.0,75.165590,0.931040,HIGH_VOLUME_WEAK_POSITION,REVIEW


In [12]:
import os

os.makedirs("/content/work/outputs", exist_ok=True)

queue.to_csv(
    "/content/work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV saved successfully.")

CSV saved successfully.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [13]:
top20 = queue.head(20).copy()

top20["confidence_note"] = "Medium confidence: simple rule-based ranking."

top20["what_would_make_it_wrong"] = (
    "Seasonality, temporary ranking changes, or other business context."
)

display(
    top20[
        [
            "rank",
            "action",
            "reason_code",
            "confidence_note",
            "what_would_make_it_wrong"
        ]
    ]
)

,rank,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,REVIEW,HIGH_VOLUME_WEAK_POSITION,Medium confidence: simple rule-based ranking.,"Seasonality, temporary ranking changes, or oth..."
1,2,REVIEW,HIGH_VOLUME_WEAK_POSITION,Medium confidence: simple rule-based ranking.,"Seasonality, temporary ranking changes, or oth..."
2,3,REVIEW,HIGH_VOLUME_WEAK_POSITION,Medium confidence: simple rule-based ranking.,"Seasonality, temporary ranking changes, or oth..."
3,4,REVIEW,HIGH_VOLUME_WEAK_POSITION,Medium confidence: simple rule-based ranking.,"Seasonality, temporary ranking changes, or oth..."
4,5,REVIEW,HIGH_VOLUME_WEAK_POSITION,Medium confidence: simple rule-based ranking.,"Seasonality, temporary ranking changes, or oth..."
5,6,REVIEW,HIGH_VOLUME_WEAK_POSITION,Medium confidence: simple rule-based ranking.,"Seasonality, temporary ranking changes, or oth..."
6,7,REVIEW,HIGH_VOLUME_WEAK_POSITION,Medium confidence: simple rule-based ranking.,"Seasonality, temporary ranking changes, or oth..."
7,8,REVIEW,HIGH_VOLUME_WEAK_POSITION,Medium confidence: simple rule-based ranking.,"Seasonality, temporary ranking changes, or oth..."
8,9,REVIEW,HIGH_VOLUME_WEAK_POSITION,Medium confidence: simple rule-based ranking.,"Seasonality, temporary ranking changes, or oth..."
9,10,REVIEW,HIGH_VOLUME_WEAK_POSITION,Medium confidence: simple rule-based ranking.,"Seasonality, temporary ranking changes, or oth..."


The top 20 pages were reviewed using their action, reason code, and confidence note. These are prioritization recommendations rather than guaranteed problems. A recommendation could be wrong because of seasonality, temporary ranking changes, search intent, or business context.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: Some recommendations may be weak because the baseline only uses impressions and average position. It does not account for seasonality, search intent, content quality, or business value.

Leakage check: The score uses only observed March 2026 search performance. It does not use future-window data, product decision flags, or label-derived features.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.